In [1]:
!pip install -q fastparquet
!pip install -q statsforecast

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 41.2 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.6/354.6 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.0/281.0 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 47.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 kB 2.7 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsforecast import StatsForecast
from statsforecast.models import TSB, AutoETS, AutoARIMA, Naive, SeasonalNaive, Theta, OptimizedTheta, CrostonOptimized, ADIDA, IMAPA, CrostonSBA, HoltWinters

%matplotlib inline


# Импортируем .py файл с уже написанными функциями
import sys
sys.path.append('/kaggle/input/datasets/faibus/diploma')

from metrics import (
    DEFAULT_METRICS,
    get_model_columns,
    compute_metrics_per_window,
    summarize_metrics,
)

# функции для фильтрации и разделения рядов
from split_utils import filter_long_series, split_train_val_test

# функции для оценки и сбора предсказаний
from evaluation_utils import evaluate_frozen_windows

In [3]:
# данные о реальном спросе
real_demand = pd.read_parquet('/kaggle/input/datasets/faibus/diploma/real_demand_data.parquet', engine='fastparquet')

# выгружаем типы рядов
ts_dict_df = pd.read_csv('/kaggle/input/datasets/faibus/diploma/ts_dict_df')[['SKU_id', 'ts_type']]

# данные о праздниках 
holidays = pd.read_excel('/kaggle/input/datasets/faibus/diploma/holidays.xlsx')
holidays['Holidays'] = pd.to_datetime(holidays['Holidays'], format='%Y.%m.%d')

# данные о погоде
weather = pd.read_csv('/kaggle/input/datasets/faibus/diploma/weather.csv')
weather['final_temp'] = (weather['temperature_ekb'] * 0.14) + (weather['temperature_msk'] * 0.22) + (weather['temperature_nov'] * 0.11) + (weather['temperature_ros'] * 0.16) + (weather['temperature_sam'] * 0.15) + (weather['temperature_spb'] * 0.09) + (weather['temperature_tve'] * 0.13)
weather = weather[['date', 'final_temp']]
weather['date'] = pd.to_datetime(weather['date'], format='%Y-%m-%d')

# данные о днях промо
promo_days = pd.read_parquet('/kaggle/input/datasets/faibus/diploma/promo_days.parquet', engine='fastparquet')

# данные о характеристиках SKU
sku_charasterictics = pd.read_parquet('/kaggle/input/datasets/faibus/diploma/sku_features.parquet', engine='fastparquet')[['SKU_id', 'SupplierID']]

In [4]:
# Выделяем SKU, принадлежащие к типу lumpy
lumpy_ts = list(ts_dict_df.query(" ts_type == 'lumpy' ")['SKU_id'])

# фильтруем датасет по lumpy_ts
real_demand = real_demand.query(" SKU_id.isin(@lumpy_ts) ")

# причесываем датасет
real_demand = real_demand.rename(columns = {'date':'ds', 'real_demand':'y', 'SKU_id':'unique_id'})[['unique_id', 'ds', 'y']]

real_demand.shape

(912713, 3)

In [5]:
# джойним характеристики SKU
real_demand = pd.merge(real_demand, sku_charasterictics, left_on = ['unique_id'], right_on = ['SKU_id'])

# создаём дни праздников (корректнее сказать "признак выходных")
real_demand['is_holiday'] = real_demand['ds'].isin(holidays['Holidays'])

# джойним промо-дни
real_demand = pd.merge(real_demand, promo_days, how = 'left', left_on = ['ds', 'unique_id'], right_on = ['date', 'SKU_id'])

# джойним погодные условия (температуру)
real_demand = pd.merge(real_demand, weather, how = 'left', left_on = ['ds'], right_on = ['date'])

# заполяем пропуски в промо-днях значением False
real_demand['is_promo'] = real_demand['is_promo'].fillna(False)

# оставляем только нужные колонки
real_demand = real_demand[['unique_id', 'ds', 'y', 'is_holiday', 'is_promo','final_temp']].copy()

# генерим временные фичи             
real_demand['dayofweek'] = real_demand['ds'].dt.dayofweek + 1  

# меняем тип данных bool -> int
real_demand['is_holiday'] = real_demand['is_holiday'].astype(int)
real_demand['is_promo'] = real_demand['is_promo'].astype(int)

real_demand.shape

/tmp/ipykernel_55/88606493.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  real_demand['is_promo'] = real_demand['is_promo'].fillna(False)


(912713, 7)

### Фильтруем ряды

In [6]:
real_demand_filtered = filter_long_series(
    real_demand,
    horizon=14,
    n_val_windows=1,
    n_test_windows=3,
    min_train_points=35,
    id_col="unique_id",
)

eval_df = real_demand_filtered[["unique_id", "ds", "y", "is_holiday", "is_promo", "final_temp", "dayofweek"]].sort_values(["unique_id", "ds"])
eval_df.shape

(881994, 7)

### Обучение

In [7]:
horizon = 14

# 1. Создаем список с моделями
models = [
    AutoARIMA(alias='AutoARIMA_exog'),
]

sf = StatsForecast(models=models, freq="D", n_jobs=-1, fallback_model=Naive())

cv_frozen = sf.cross_validation(
    df = eval_df,
    h = 14,
    step_size = 14,            
    n_windows = 4,
    refit = False                      
)

### Собираем прогнозы

In [8]:
val_cutoff = pd.Timestamp("2025-08-05")
test_cutoffs = [
    pd.Timestamp("2025-08-19"),
    pd.Timestamp("2025-09-02"),
    pd.Timestamp("2025-09-16"),
]

val_pred = cv_frozen[cv_frozen["cutoff"] == val_cutoff].copy()
test_pred = cv_frozen[cv_frozen["cutoff"].isin(test_cutoffs)].copy()

cutoff_to_window = {c: i + 1 for i, c in enumerate(test_cutoffs)}
test_pred["test_window"] = test_pred["cutoff"].map(cutoff_to_window)

### Считаем метрики

In [9]:
from metrics import DEFAULT_METRICS, get_model_columns, compute_metrics_per_window, summarize_metrics

val_model_cols = get_model_columns(
    val_pred,
    reserved_columns=("unique_id", "ds", "cutoff", "y", "index", "test_window"),
)
val_metrics_per_window = compute_metrics_per_window(val_pred, val_model_cols, DEFAULT_METRICS)
val_summary_mean, val_summary_stats = summarize_metrics(val_metrics_per_window)

test_model_cols = get_model_columns(
    test_pred,
    reserved_columns=("unique_id", "ds", "cutoff", "y", "index", "test_window"),
)
test_metrics_per_window = compute_metrics_per_window(test_pred, test_model_cols, DEFAULT_METRICS)
test_summary_mean, test_summary_stats = summarize_metrics(test_metrics_per_window)

print("VAL mean metrics:")
display(val_summary_mean)

print('')

print("TEST mean metrics (3 windows x 14 days):")
display(test_summary_mean)

VAL mean metrics:


,mae,rmse,smape,wape
model,,,,
AutoARIMA_exog,1.132159,2.326877,152.461579,105.928618



TEST mean metrics (3 windows x 14 days):


,mae,rmse,smape,wape
model,,,,
AutoARIMA_exog,1.236571,2.963294,156.507089,111.858468


In [10]:
test_pred.to_csv("arima_lumpy.csv")